# test_revised triple 추출 — 최종 모델(DREAM revised)

최종 채택 모델 `atlop_dream_revised.pt`로 `test_revised.json`의 관계 triple을 추출한다.
(test_revised에 gold labels가 있어 F1도 함께 계산)

**실행 전**: `Scripts/atlop/extract_triples.py`·`re_model_dream.py`·`docred_io.py`가 **dh에 push**돼 있어야 하고,
**학습된 `atlop_dream_revised.pt`가 Drive에** 있어야 함.


## 1) Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) 코드(dh) + revised 데이터(main, LFS) 가져오기

In [ ]:
%cd /content
!apt-get -qq install -y git-lfs > /dev/null 2>&1
!git lfs install --skip-repo
!rm -rf project1
!git clone -q https://github.com/multiful/Information_Extraction.git project1
%cd /content/project1
!git checkout -q dh
# test_revised(추출 대상) + train_revised(Ign F1 필터용) 실체화
!git checkout origin/main -- docred_data/data/test_revised.json docred_data/data/train_revised.json
!git lfs pull --include="docred_data/data/test_revised.json,docred_data/data/train_revised.json"
!pip install -q transformers==4.57.6 accelerate

import re, pathlib
_p = pathlib.Path("data/docred_io.py"); _s = _p.read_text(encoding="utf-8")
if "test_revised" not in _s:
    _p.write_text(re.sub(r'SPLITS\s*=\s*\[[^\]]*\]',
        'SPLITS = ["train_annotated","train_distant","dev","test","train_revised","dev_revised","test_revised"]',
        _s, count=1), encoding="utf-8")
    print("docred_io SPLITS 런타임 패치")

import os
assert os.path.exists("Scripts/atlop/extract_triples.py"), "extract_triples.py 없음 -> dh push 확인!"
for f in ["test_revised", "train_revised"]:
    kb = os.path.getsize(f"docred_data/data/{f}.json") // 1024
    print(f"{f}: {kb} KB", "OK" if kb > 100 else "  <-- LFS 실체화 실패")

## 3) 인코더(bert-base-cased) 로컬 다운로드

In [ ]:
import os
os.makedirs("/content/bert-base-cased", exist_ok=True)
%cd /content/bert-base-cased
B  = "https://huggingface.co/google-bert/bert-base-cased/resolve/main"
UA = "Mozilla/5.0"
for f in ["config.json", "vocab.txt", "tokenizer.json", "tokenizer_config.json", "model.safetensors"]:
    print("↓", f, flush=True)
    !wget -c --tries=200 --timeout=30 --waitretry=3 --header="User-Agent: {UA}" -O {f} {B}/{f}
%cd /content/project1
!ls -lh /content/bert-base-cased/model.safetensors

## 4) triple 추출 (+ test F1)
- `--ckpt` : Drive의 학습된 체크포인트 경로 (본인 경로로!)
- `--split test_revised` / `--eval` : gold labels로 F1도 계산 (Ign 필터 = train_revised)
- 결과: `results/atlop_dream_revised_test_revised_triples.json` (문서별 triple)


In [ ]:
CKPT = "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/atlop_dream_revised.pt"  # <- 본인 경로

!python -u -m Scripts.atlop.extract_triples \
  --ckpt "{CKPT}" \
  --model_name_or_path /content/bert-base-cased \
  --split test_revised --eval --eval_batch_size 32 \
  --out results/atlop_dream_revised_test_revised_triples.json

## 5) 결과 미리보기 + Drive 백업

In [ ]:
import json
tp = json.load(open("results/atlop_dream_revised_test_revised_triples.json", encoding="utf-8"))
print("총 triple:", len(tp), "| 문서 수:", len({t["title"] for t in tp}))
for t in tp[:10]:
    print(f'  ({t["head"]}) -[{t["relation"]}]-> ({t["tail"]})   [{t["title"]}]')

dst = "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"   # <- 본인 경로
!cp results/atlop_dream_revised_test_revised_triples.json "{dst}/"
print("✅ Drive 저장 완료")